# Advanced RAG and evaluation

Assistant version 5: Compare baseline and improved retrieval on a held-out golden set.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra rag --group authoring`, then `uv run --extra rag --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Query rewriting; multi-query retrieval; deduplication

Predict the output before running. The function definitions below match `examples/01_query_rewrite.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Generate bounded retrieval variants, retrieve each, and deduplicate by chunk identity.

Run: uv run --extra rag python modules/09_advanced_rag/examples/01_query_rewrite.py --offline
Expected: Standalone variants and a deduplicated evidence list.
Cost: One rewrite call; offline fixtures are not semantic query understanding.
Requires: rag
"""

from common.advanced_rag import rewrite
from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import LexicalRetriever, fuse


def main() -> None:
    # KEY IDEA: Retain the original query so a mistaken rewrite cannot erase its retrieval path.
    settings = lesson_settings()
    queries = rewrite("How long is manufacturing defect coverage?", settings)
    retriever = LexicalRetriever(chunk_documents(policy_documents()))
    found = fuse([retriever.search(query) for query in queries])
    print_text(str(queries))
    print_text(str([d.id for d in found]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## BM25 plus vector search; reciprocal-rank fusion by hand

Predict the output before running. The function definitions below match `examples/02_hybrid.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Fuse BM25 and vector rankings by rank rather than incompatible scores.

Run: uv run --extra rag python modules/09_advanced_rag/examples/02_hybrid.py --offline
Expected: A deduplicated hybrid list; fake-vector ranking is explicitly non-semantic.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import LexicalRetriever, fuse, memory_index


def main() -> None:
    settings = lesson_settings()
    chunks = chunk_documents(policy_documents())
    lexical, vectors = LexicalRetriever(chunks), memory_index(chunks, settings)
    question = "unused gear return time limit"
    # KEY IDEA: reciprocal rank fusion uses 1/(60+rank), not addition of raw distances.
    results = fuse([lexical.search(question), vectors.similarity_search(question, k=3)])
    print_text(str([d.metadata for d in results]))
    if settings.course_offline:
        print_text(
            "Fake-vector fusion verifies mechanics only; the benchmark excludes fake vectors."
        )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Reranking; contextual compression using model scoring and bounded extracts

Predict the output before running. The function definitions below match `examples/03_rerank_compress.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Score a bounded candidate set and validate model-selected verbatim extracts.

Run: uv run --extra rag python modules/09_advanced_rag/examples/03_rerank_compress.py --offline
Expected: At most two relevant extracts, each copied from its original evidence.
Cost: Up to four reranking calls here; helper caps candidates at six. Offline is free.
Requires: rag
"""

from common.advanced_rag import rerank_compress
from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import LexicalRetriever


def main() -> None:
    # KEY IDEA: Bound reranking calls and reject extracts that change the source text.
    settings = lesson_settings()
    question = "When is manager approval required?"
    candidates = LexicalRetriever(chunk_documents(policy_documents())).search(question, k=4)
    for doc in rerank_compress(question, candidates, settings):
        print_text(doc.page_content)
    print_text("Exact extraction protects provenance; it does not prove semantic relevance.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Parent-document small-to-big retrieval implemented with parent IDs

Predict the output before running. The function definitions below match `examples/04_parent_retrieval.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Retrieve small children, then expand them to original parent documents.

Run: uv run --extra rag python modules/09_advanced_rag/examples/04_parent_retrieval.py
--offline
Expected: Child IDs followed by fewer, larger parent documents.
Cost: No model calls or API fees.
Requires: rag
"""

from common.advanced_rag import expand_parents
from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import LexicalRetriever


def main() -> None:
    lesson_settings()
    parents = policy_documents()
    children = chunk_documents(parents, size=160, overlap=30)
    found = LexicalRetriever(children).search("refund manager approval")
    expanded = expand_parents(found, parents)
    # KEY IDEA: parent IDs map back to unchanged originals, not reconstructed text fragments.
    print_text(str([doc.id for doc in found]))
    print_text(str([(doc.metadata["source"], len(doc.page_content)) for doc in expanded]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Structured query extraction; validated metadata filters

Predict the output before running. The function definitions below match `examples/05_query_filters.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Translate a constrained query into a validated metadata filter.

Run: uv run --extra rag python modules/09_advanced_rag/examples/05_query_filters.py --offline
Expected: Only four-person tents under $300; no generated SQL is executed.
Cost: One extraction call plus product embeddings; offline is free.
Requires: rag
"""

from common.advanced_rag import extract_filter
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import product_documents
from common.vector_index import persistent_index


def main() -> None:
    settings = lesson_settings()
    spec = extract_filter("Show four-person tents under $300", settings)
    where = {
        "$and": [
            {"category": spec.category},
            {"capacity": spec.capacity},
            {"price": {"$lt": spec.max_price}},
        ]
    }
    store = persistent_index(product_documents(), settings, name="lesson09-filters")
    # KEY IDEA: only allowlisted fields/operators are assembled by application code.
    print_text(str(where))
    print_text(
        str([doc.metadata for doc in store.similarity_search("waterproof tent", k=3, filter=where)])
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## History-aware rewriting of follow-up questions

Predict the output before running. The function definitions below match `examples/06_conversation.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Resolve a follow-up question from explicitly supplied conversation history.

Run: uv run --extra rag python modules/09_advanced_rag/examples/06_conversation.py --offline
Expected: Standalone warranty queries and a cited answer; no memory is stored automatically.
Cost: One rewrite and one answer call; offline fixtures have no API fee.
Requires: rag
"""

from common.advanced_rag import rewrite
from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.rag import answer_question
from common.retrieval import LexicalRetriever, fuse


def main() -> None:
    # KEY IDEA: Explicit history resolves references; retrieved evidence supplies the facts.
    settings = lesson_settings()
    history = [
        {"role": "user", "content": "I have a warranty question."},
        {
            "role": "assistant",
            "content": "What would you like to know about manufacturing defects?",
        },
    ]
    question = "How long does it last?"
    queries = rewrite(question, settings, history)
    print_text(str(queries))
    retriever = LexicalRetriever(chunk_documents(policy_documents()))
    evidence = fuse([retriever.search(query) for query in queries])
    # The standalone query supplies the subject that the latest question leaves implicit.
    print_text(str(answer_question(queries[-1], evidence, settings)))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Retrieval hit rate, faithfulness, answer correctness; local judge and optional LLM judge

Predict the output before running. The function definitions below match `examples/07_evaluate.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Separate retrieval hits, answer proxies and optional model-judge verdicts.

Run: uv run --extra rag python modules/09_advanced_rag/examples/07_evaluate.py --offline
Expected: A local metric report and a clearly labelled scripted judge verdict.
Cost: One answer and one judge call; offline judge is scripted and is not scored as quality.
Requires: rag
"""

from common.data import load_jsonl
from common.errors import run_cli
from common.evaluation import answer_metrics, judge_answer, retrieval_hit
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.rag import answer_question, context_text
from common.retrieval import LexicalRetriever


def main() -> None:
    # KEY IDEA: Score retrieval and answers separately; scripted judges cannot assess meaning.
    settings = lesson_settings()
    case = load_jsonl("eval/rag.jsonl")[0]
    found = LexicalRetriever(chunk_documents(policy_documents())).search(case["question"])
    answer = answer_question(case["question"], found, settings)
    print_text(str({"retrieval_hit": retrieval_hit(case, found), **answer_metrics(case, answer)}))
    verdict = judge_answer(case["question"], answer, case["answer"], context_text(found), settings)
    print_text(
        (
            "FIXTURE judge: "
            if settings.course_offline
            else "LIVE model judge (needs human calibration): "
        )
        + verdict.model_dump_json()
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Missing evidence, stale indexes, bad chunks, injection and citation mismatch; before/after results

Predict the output before running. The function definitions below match `examples/08_failure_gallery.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Show missing evidence, bad chunks, poisoned text, stale identity and invalid citations.

Run: uv run --extra rag python modules/09_advanced_rag/examples/08_failure_gallery.py --offline
Expected: Five failure labels with concrete checks and fixes.
Cost: No model calls or API fees.
Requires: rag
"""

from common.data import data_path
from common.errors import run_cli
from common.knowledge import LocalLoader, chunk_documents, html_document
from common.lesson import lesson_settings
from common.pretty import print_text
from common.rag import Citation, GroundedAnswer, extractive_fixture, validate_citations
from common.vector_index import embedding_identity


def main() -> None:
    # KEY IDEA: Each failure needs its own check; removing HTML comments is not full protection.
    settings = lesson_settings()
    docs = LocalLoader("policies/returns.md").load()
    print_text("Missing evidence: " + str(extractive_fixture("Unknown award?", [])))
    print_text(
        "Bad chunks: "
        + str([d.page_content for d in chunk_documents(docs, size=25, overlap=0)][:3])
    )
    poison = data_path("security/poisoned_doc.md").read_text(encoding="utf-8")
    print_text(
        "Injection: comments removed from HTML extraction, but this is NOT a complete defense."
    )
    assert "attacker@example.test" not in html_document(poison, "synthetic-poison").page_content
    identity = embedding_identity(settings, 64)
    print_text(f"Stale index: identity {identity} must match the stored manifest before reopening.")
    try:
        validate_citations(
            GroundedAnswer(
                answer="Unsupported",
                abstained=False,
                citations=[Citation(chunk_id=str(docs[0].id), quote="invented quote")],
            ),
            docs,
        )
    except ValueError as error:
        print_text("Citation mismatch: " + str(error))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Compare baseline and improved retrieval on a held-out golden set

Predict the output before running. The function definitions below match `use_case/rag_benchmark.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare baseline and candidate retrieval without inventing a quality gain.

Run: uv run --extra rag python modules/09_advanced_rag/use_case/rag_benchmark.py --offline
Expected: A saved before/after report; identical hit rates are reported honestly.
Cost: Offline no API calls. Live: up to 16 generations plus embeddings, budget $0.10.
Requires: rag
"""

from common.config import ROOT
from common.errors import run_cli
from common.evaluation import benchmark, save_report
from common.lesson import lesson_settings
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    report = benchmark(settings)
    output = ROOT / settings.runtime_dir / "rag_benchmark.json"
    save_report(report, output)
    for name, metrics in report["systems"].items():
        print_text(
            str({"system": name, **{key: value for key, value in metrics.items() if key != "rows"}})
        )
    print_text(str(output))
    print_text(report["limits"])


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra rag pytest modules/09_advanced_rag/tests` for the supplied solution checks.